[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/foucault/foucault.ipynb)

# Foucault's Pendulum

This example computes the daily precession of Foucault's pendulum and its spherical parallel-transport holonomy across northern latitudes. In 1851 Léon Foucault hung a heavy bob on a long wire from the dome of the Panthéon in Paris, and as the hours passed its vertical plane of oscillation turned steadily clockwise seen from above without any lateral force on the bob. Because gravity and wire tension act purely in the plane of the vertical, nothing twists the swing within the horizontal floor while the spinning earth carries the floor around its axis. Only the component of the earth's turn lying in the floor twists the room beneath the bob, reaching a full turn per day at the pole and vanishing at the equator, and carrying a tangent direction around the latitude circle without horizontal twist reproduces the same daily angle from the solid angle of the enclosed polar cap.

Both the earth's rotation and the site's floor are modelled directly as bivector planes, `equator` and the commutator `east.commutator(north)` of the local compass directions, so the fraction of the earth's turn felt by the floor is the plane-to-plane projection `(equator | floor) / floor.squared()` with no axial rotation vector in between. Around the latitude circle, the minimal rotor carrying one site's vertical onto the next is `(1 + sites[1:] * ~sites[:-1]).normalized()`, and multiplying those step rotors along the loop yields the holonomy rotor directly.

In the Gibbs–Heaviside vector notation of rotating frames, transforming into the earth-fixed frame introduces the fictitious Coriolis acceleration $-2\,\boldsymbol{\Omega}\times\dot{\mathbf{r}}$ from the axial angular-velocity vector $\boldsymbol{\Omega}$, whose horizontal projection couples the local $(x, y)$ coordinates through $\ddot{x} - 2\Omega\sin\varphi\,\dot{y} + \omega_0^2 x = 0$ and $\ddot{y} + 2\Omega\sin\varphi\,\dot{x} + \omega_0^2 y = 0$ at latitude $\varphi$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.util.animation import show_states
from examples.mechanics.foucault import render, scenarios

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
# Spatial directions carry intrinsic antivector orientation.
Point = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()

# Coordinate planes: x and y meet along the rotation axis, x passes through the prime meridian, and z is the equator.
x, y, z = mv.basis()                                                          # [] Plane+, Plane+, Plane+
# Equatorial plane in which the earth turns.
equator = mv.xy                                                               # [] Bivector+

## Projection of the earth's turn onto the floor

At any site on the globe, the local north and east directions span the horizontal floor plane via `east.commutator(north)`, while the earth turns in `equator`. Taking the inner product of those two bivectors over `floor.squared()` projects one rotation plane onto the other and gives the sine of the latitude: unity at the pole, three quarters at Paris, one half at thirty degrees, and zero at the equator. Relative to the turning room, a swing started along north rotates backward within the floor by that share of the earth's turn.

In [ ]:
n_latitudes = scenarios.n_latitudes
latitudes = scenarios.latitudes                                               # [n_latitudes]
paris = scenarios.paris

In [ ]:
def compass(latitudes: np.ndarray) -> tuple[Point, Point, Point]:
    """Local vertical, north, and east (`[...] Point-`) frame each site on the prime meridian at `latitudes`."""
    # Tilt the prime-meridian plane normal toward the north pole by the latitude, and a quarter turn further for north.
    up_turn = (mv.xz * (-latitudes / 2)).exp()                                # [...] Rotor+
    north_turn = (mv.xz * (-(latitudes + np.pi / 2) / 2)).exp()               # [...] Rotor+
    vertical = up_turn >> x.dual()                                            # [...] Point-
    north = north_turn >> x.dual()                                            # [...] Point-
    return vertical, north, y.dual() * np.ones_like(latitudes)                # [...] Point-, Point-, Point-


def floor_share(floors: Bivector) -> Scalar:
    """Orthogonal projection of `equator` onto `floors` measures the fraction (`[...] Scalar+`) of the earth's turn felt in each floor."""
    return (equator | floors) / floors.squared()                              # [...] Scalar+


_, north, east = compass(latitudes)                                           # [n_latitudes] Point-, Point-, Point-
floors = east.commutator(north)                                               # [n_latitudes] Bivector+
print("fraction of the earth's turn felt in each floor:", (-floor_share(floors)).to_array().round(4) + 0.0)

In [ ]:
def swing_direction(latitudes: np.ndarray, earth_turns: np.ndarray) -> Point:
    """Starting along north, each pendulum swing (`[...] Point-`) turns backward within its floor by `floor_share` of `earth_turns`."""
    _, north, east = compass(latitudes)                                       # [...] Point-, Point-, Point-
    # Orthogonal directions anticommute, so their commutator spans the oriented horizontal floor plane.
    floor = east.commutator(north)                                            # [...] Bivector+
    swing_turn = (floor * floor_share(floor) * (earth_turns / 2)).exp()       # [...] Rotor+
    return swing_turn >> north                                                # [...] Point-


full_day = np.full(n_latitudes, 2 * np.pi)                                    # [n_latitudes]
day_swings = swing_direction(latitudes, full_day)                             # [n_latitudes] Point-

## Parallel transport around the latitude circle

Carrying a floor direction around the latitude circle in small steps, where each step applies the minimal rotor taking one site's vertical to the next without twisting about the vertical, brings the direction back turned after one circuit. That accumulated turn equals a full rotation minus the solid angle of the polar cap bounded by the latitude circle, matching the pendulum's clockwise daily precession.

In Riemannian surface geometry, the Levi-Civita connection along a parallel of colatitude $\theta = \frac{\pi}{2} - \varphi$ integrates the geodesic curvature $\kappa_g = \cot\theta$ over circumference $2\pi\sin\theta$, giving the Gauss–Bonnet holonomy angle $2\pi - \iint_{\text{cap}} K\,dA = 2\pi(1 - \sin\varphi)$.

In [ ]:
n_steps = scenarios.n_steps

In [ ]:
def parallel_holonomy(latitudes: np.ndarray, n_steps: int) -> Rotor:
    """Minimal rotors between `n_steps` successive verticals around each latitude circle compose into the parallel-transport holonomy (`[...] Rotor+`)."""
    vertical, _, _ = compass(latitudes)                                       # [...] Point-, Point-, Point-
    # Carry the site's vertical through one full rotation of the earth in n_steps equal increments.
    day = (equator * (-np.pi * np.arange(n_steps + 1)[:, None] / n_steps)).exp()  # [n_steps + 1, ...] Rotor+
    sites = day >> vertical                                                   # [n_steps + 1, ...] Point-
    steps = (1 + sites[1:] * ~sites[:-1]).normalized()                        # [n_steps, ...] Rotor+
    return steps.cumprod(axis=0)[-1]                                          # [...] Rotor+


def turn_angle(rotors: Rotor, floors: Bivector) -> Scalar:
    """Pairing twice the bivector logarithm of `rotors` with `floors` reads the clockwise turn angle (`[...] Scalar+`) in radians."""
    return 2 * (rotors.log() | floors)                                        # [...] Scalar+


holonomy = parallel_holonomy(latitudes, n_steps)                              # [n_latitudes] Rotor+
transported = holonomy >> north                                               # [n_latitudes] Point-
day_rotors = (floors * floor_share(floors) * np.pi).exp()                     # [n_latitudes] Rotor+
print("holonomy clockwise turn (deg):", (turn_angle(holonomy, floors) * (180 / np.pi)).to_array().round(2) % 360)
print("pendulum daily turn (deg):    ", (turn_angle(day_rotors, floors) * (180 / np.pi)).to_array().round(2) % 360)

## Daily precession in Paris

As the globe completes one sidereal rotation, the Panthéon floor rides eastward around its latitude circle along the rim of the polar cap while the pendulum's swing rod turns steadily clockwise relative to the floor's north rod.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def day_states(latitude: float, n_frames: int) -> Iterator[tuple[Rotor, Point]]:
    """Yields `(earth, swing)` (`[] Rotor+`, `[] Point-`) across `n_frames` equal intervals of one sidereal day at `latitude`."""
    _, north, east = compass(np.asarray(latitude))                            # [] Point-, Point-, Point-
    floor = east.commutator(north)                                            # [] Bivector+
    share = floor_share(floor)                                                # [] Scalar+
    for index in range(n_frames + 1):
        angle = 2 * np.pi * index / n_frames
        earth = (equator * (-angle / 2)).exp()                                # [] Rotor+
        swing_turn = (floor * share * (angle / 2)).exp()                      # [] Rotor+
        yield earth, swing_turn >> north


paris_vertical, paris_north, paris_east = compass(np.asarray(latitudes[paris]))  # [] Point-, Point-, Point-
recorded_day = show_states(
    day_states(float(latitudes[paris]), n_frames),
    render.day_animation,
    paris_vertical,
    paris_north,
    paris_east,
    name="foucault_day",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

The blue polar cap and latitude circle mark the solid angle enclosed north of Paris, the grey circle marks the equator, the dark rod extending past the cream floor slab points along local north, and the orange rod marks the line of swing, completing three quarters of a clockwise turn relative to north over the day.

In [ ]:
# checks
# After one sidereal day each swing turns clockwise by sin(latitude) of a full turn, matching parallel transport.
expected = (floors * (-np.pi * np.sin(latitudes))).exp() >> north             # [n_latitudes] Point-
assert ((day_swings - expected).antinorm() < 1e-10).all()
assert ((transported - expected).antinorm() < 1e-4).all()
final_earth, final_swing = recorded_day[-1]                                   # [] Rotor+, [] Point-
assert (final_earth >> paris_north - paris_north).antinorm() < 1e-10
assert (final_swing - expected[paris]).antinorm() < 1e-10